# JavaScript kernel comm spike (A2UI feasibility)

Runs on `@jupyterlite/javascript-kernel@0.4.0-alpha.5`. Checks:

- **A**: the `jupyter.widget` comm displays a widget.
- **B**: an awaited cell resolves from a frontend `comm_msg`.
- **D**: frontend messages and kernel state updates work without blocking a cell.

In [ ]:
const esm = `
export default {
  render({ model, el }) {
    const btn = document.createElement('button');
    btn.id = 'spike-go';
    btn.textContent = 'Go';
    btn.onclick = () => model.send({ action: 'go' });
    const out = document.createElement('div');
    out.id = 'spike-out';
    const paint = () => { out.textContent = 'count=' + model.get('count'); };
    model.on('change:count', paint);
    paint();
    el.append(btn, out);
  }
};`;

globalThis.spikeComm = Jupyter.comm.open('jupyter.widget', {
  state: {
    _model_name: 'AnyModel', _model_module: 'anywidget', _model_module_version: '0.11.0',
    _view_name: 'AnyView', _view_module: 'anywidget', _view_module_version: '0.11.0',
    _esm: esm, _css: '', count: 0
  },
  buffer_paths: []
}, {version: '2.1.0'}, [], 'spike-1');
spikeComm.display();
'opened'

In [ ]:
const reply = await new Promise((resolve) => {
  spikeComm.onMsg = (data) => resolve(data);
});
console.log('B reply ' + JSON.stringify(reply));
reply

In [ ]:
globalThis.spikeLog = [];
spikeComm.onMsg = (data) => { globalThis.spikeLog.push(data); };
spikeComm.send({ method: 'update', state: { count: 5 }, buffer_paths: [] });
'sent update'

In [ ]:
console.log('D log ' + JSON.stringify(globalThis.spikeLog));
globalThis.spikeLog